# aerial-vg: TransVG on DIOR-RSVG + direction-aware D4 augmentation

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. The dataset is downloaded from the authors' Google Drive into `/tmp` (~5 GB).

`MODE` in the config cell selects what runs (two runs in parallel, one per GPU, in two phases):

| MODE | phase 1 | phase 2 | total |
|---|---|---|---|
| `main` (report, Kaggle Version 5) | baseline ‖ d4_remap (seed 13) | d4_noremap ‖ no_geo (seed 13) | ~7.3 h |
| `seeds` | baseline_s14 ‖ d4_remap_s14 | baseline_s15 ‖ d4_remap_s15 | ~7.3 h |

The pre-LN / warm-up / clipping fix is part of the code (`--vl_prenorm 1 --warmup_iters 500 --clip_max_norm 0.1` in `scripts/run_one.sh`).
The Version 5 run applied the same patch from a notebook cell; that executed notebook is `notebooks/aerial-vg-experiments_v5_as_run.ipynb`.

Use **Save Version → Save & Run All (Commit)**. Set `SMOKE = True` for a ~20-minute end-to-end check.

In [ ]:
import time; NB_START = time.time()   # used to keep the whole run inside Kaggle's 12 h limit
REPO_URL = 'https://github.com/HonourGok/aerial-vg.git'   # used only if the code is not attached as a Kaggle dataset
import os, subprocess, glob, shutil
DATA_ROOT = '/tmp/DIOR_RSVG'   # /tmp is not saved as notebook output, so the 5 GB stay out of the 20 GB output limit

def find_dir(root, ext):
    best = None
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith(ext) for f in files)
        if n and (best is None or n > best[1]):
            best = (d, n)
    return best

if not os.path.isdir(DATA_ROOT + '/Annotations'):
    # 1) a manually uploaded Kaggle dataset, if one was added as input
    up = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/Annotations', recursive=True)]
    if up:
        DATA_ROOT = up[0]
    else:
        # 2) download from the authors' Google Drive folder (Annotations.zip 7 MB, JPEGImages.zip 5 GB)
        subprocess.run('pip -q install gdown', shell=True, check=True)
        os.makedirs('/tmp/rsvg_dl', exist_ok=True)
        subprocess.run('gdown --folder https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_ -O /tmp/rsvg_dl',
                       shell=True, check=True)
        os.makedirs(DATA_ROOT, exist_ok=True)
        for z in glob.glob('/tmp/rsvg_dl/**/*.zip', recursive=True):
            subprocess.run(f'md5sum "{z}" | tee -a /tmp/data_md5.txt', shell=True)   # data version
            subprocess.run(f'unzip -q -o "{z}" -d /tmp/rsvg_unz', shell=True, check=True)
            os.remove(z)
        for sub, ext in (('Annotations', '.xml'), ('JPEGImages', '.jpg')):
            d, n = find_dir('/tmp/rsvg_unz', ext)
            print(sub, '<-', d, n, 'files')
            shutil.move(d, f'{DATA_ROOT}/{sub}')

print('DATA_ROOT =', DATA_ROOT, '| xml:', len(os.listdir(DATA_ROOT + '/Annotations')),
      '| jpg:', len(os.listdir(DATA_ROOT + '/JPEGImages')))
# If gdown fails with a quota / "too many users" error: download the two zips manually and upload them as a
# Kaggle dataset (Datasets -> New Dataset), add it as input, and re-run this cell.

In [ ]:
%cd /kaggle/working
# code: from an attached Kaggle dataset containing aerial-vg/ (works while the GitHub repo is private), else git clone
code_src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/aerial-vg/train.py', recursive=True)]
!rm -rf aerial-vg
if code_src:
    shutil.copytree(code_src[0], '/kaggle/working/aerial-vg')
    print('code from', code_src[0])
else:
    subprocess.run(f'git clone -q {REPO_URL} aerial-vg', shell=True, check=True)
%cd /kaggle/working/aerial-vg
!(git rev-parse HEAD 2>/dev/null || md5sum train.py datasets/geo_aug.py) | tee results_commit.txt
!pip -q install -r requirements.txt
!mkdir -p checkpoints && wget -q -nc -O checkpoints/detr-r50-e632da11.pth https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -V && python -c "import torch,torchvision,transformers;print(torch.__version__,torchvision.__version__,transformers.__version__)" 

In [ ]:
# one-off: resize images to 640 px (the model input size) so the data loader does not decode 800 px JPEGs every epoch
if not os.path.isdir('/tmp/DIOR_RSVG_640/JPEGImages'):
    !python tools/preresize.py --src $DATA_ROOT --dst /tmp/DIOR_RSVG_640 --size 640
DATA_ROOT = '/tmp/DIOR_RSVG_640'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['DETR'] = '/kaggle/working/aerial-vg/checkpoints/detr-r50-e632da11.pth'
# compute budget (identical for all conditions)
# measured on T4: 17.5 train samples/s -> ~26 min per epoch on the full train split, ~5 min for the full test set
MODE = 'main'    # 'main' = the 4 conditions of the report, 'seeds' = baseline + d4_remap with seeds 14 and 15
SMOKE = False   # True = 2-minute runs on tiny data, to check the whole notebook end to end in Save & Run All
if SMOKE:
    os.environ.update(EPOCHS='1', LR_DROP='1', FRACTION='0.005', VAL_MAX='40', TEST_LIMIT='200', LAT='20', MAX_HOURS='1')
else:
    os.environ.update(EPOCHS='7', LR_DROP='5', FRACTION='1.0', VAL_MAX='500', TEST_LIMIT='0', LAT='200', MAX_HOURS='5.0')
os.environ.update(BS='8', IMSIZE='640', SEED='13')
# time budget: everything must finish inside Kaggle's 12 h limit and the remaining GPU quota
RUN_BUDGET_H = 9.3
if not SMOKE: os.environ['MAX_HOURS'] = '4.2'
print('SMOKE =', SMOKE, {k: os.environ[k] for k in ('EPOCHS', 'LR_DROP', 'FRACTION', 'VAL_MAX', 'TEST_LIMIT', 'MAX_HOURS')})
!python tools/dataset_stats.py --data_root $DATA_ROOT --split_root splits --out results/dataset_stats.json

In [ ]:
import json
# Data checks (no training): data version and image overlap between splits.
# The split files index expressions; several expressions can share one image.
import xml.etree.ElementTree as ET
if os.path.exists('/tmp/data_md5.txt'): print(open('/tmp/data_md5.txt').read())
objs = []   # running object index in sorted XML order, same as datasets/dior_rsvg.py
for f in sorted(os.listdir(os.environ['DATA_ROOT'] + '/Annotations')):
    r = ET.parse(os.path.join(os.environ['DATA_ROOT'], 'Annotations', f)).getroot()
    objs += [r.find('filename').text] * len(r.findall('object'))
imgs = {s: {objs[int(l.split()[0])] for l in open(f'splits/{s}.txt') if l.strip()} for s in ('train', 'val', 'test')}
ov = {'objects': len(objs), **{s: len(v) for s, v in imgs.items()},
      'train&val': len(imgs['train'] & imgs['val']), 'train&test': len(imgs['train'] & imgs['test']), 'val&test': len(imgs['val'] & imgs['test'])}
print(ov); os.makedirs('results', exist_ok=True); json.dump(ov, open('results/split_image_overlap.json', 'w'), indent=1)

In [ ]:
# Speed check (~3 min, interactive sessions only; skipped in Save & Run All)
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') != 'Batch':
    !CUDA_VISIBLE_DEVICES=0 python tools/benchmark.py --dataset dior_rsvg --data_root $DATA_ROOT --split_root splits \
        --detr_model $DETR --imsize 640 --max_query_len 40 --batch_size 8 --amp --num_workers 4 --iters 40 \
        --aug_crop --aug_scale --aug_translate --geo_aug d4_remap --train_fraction 0.5 --val_max 500 2>&1 | grep -v Warning

In [ ]:
# Phase 1 (one run per GPU)
# Jobs are started with subprocess.Popen and waited on directly; a failing job stops the cell within ~20 s.
import sys, importlib
sys.path.insert(0, 'scripts'); import launch; importlib.reload(launch)
if MODE == 'main':
    PHASES = [('13', [('baseline', 'hflip', 0), ('d4_remap', 'd4_remap', 1)]),
              ('13', [('d4_noremap', 'd4_noremap', 0), ('no_geo', 'none', 1)])]
else:
    PHASES = [('14', [('baseline_s14', 'hflip', 0), ('d4_remap_s14', 'd4_remap', 1)]),
              ('15', [('baseline_s15', 'hflip', 0), ('d4_remap_s15', 'd4_remap', 1)])]
os.environ['SEED'] = PHASES[0][0]
# strict (stop on first failure) only in SMOKE; in the long run a failure of one job must not kill the other
launch.run_parallel(PHASES[0][1], strict=SMOKE)

In [ ]:
# Phase 2. Time guard: runs stop after the epoch that crosses MAX_HOURS (~0.5 h per epoch), then are evaluated (~0.2 h).
elapsed = (time.time() - NB_START) / 3600
if not SMOKE:
    os.environ['MAX_HOURS'] = '%.2f' % max(0.5, min(5.0, RUN_BUDGET_H - elapsed - 0.5 - 0.3))
print('elapsed %.2f h, phase-2 MAX_HOURS = %s' % (elapsed, os.environ['MAX_HOURS']))
os.environ['SEED'] = PHASES[1][0]
launch.run_parallel(PHASES[1][1], strict=SMOKE)

In [ ]:
# Collect: comparison over the runs that produced metrics, figure, training logs, zip
ALL = [n for _, jobs in PHASES for n, _, _ in jobs]
runs = [r for r in ALL if os.path.exists(f'results/{r}/metrics.json')]
print('runs with results:', runs)
if MODE == 'main' and 'baseline' in runs and len(runs) > 1:
    subprocess.run(['python', 'compare.py', '--runs'] + [f'{r}=results/{r}' for r in runs] +
                   ['--ref', 'baseline', '--out', 'results/comparison'], check=False)
if MODE == 'main' and 'baseline' in runs and 'd4_remap' in runs:
    subprocess.run(['python', 'tools/viz_examples.py', '--n', '8', '--a', 'results/baseline', '--b', 'results/d4_remap',
                    '--data_root', DATA_ROOT, '--out', 'results/figures/baseline_vs_d4remap.png'], check=False)
for r in ALL:
    if os.path.exists(f'outputs/{r}/log.txt'):
        os.makedirs(f'results/{r}', exist_ok=True)
        shutil.copy(f'outputs/{r}/log.txt', f'results/{r}/train_log.txt')
os.makedirs('results/logs', exist_ok=True)
for f in glob.glob('logs/*.log'):
    shutil.copy(f, 'results/logs/')
shutil.copy('results_commit.txt', 'results/')
if MODE == 'seeds':   # mean/sd over seeds 13-15 (seed 13 = the main run, read from results/ in the repo if present)
    subprocess.run(['python', 'tools/seed_summary.py', '--results', 'results'], check=False)
print('total hours: %.2f' % ((time.time() - NB_START) / 3600))
!cd /kaggle/working && rm -f results.zip && zip -qr results.zip aerial-vg/results && ls -lh results.zip
!du -sh /kaggle/working/aerial-vg/outputs